In [ ]:
import os
import random
import logging
import datetime
import argparse
import ast
import sys
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
from huggingface_hub import login
from collections import Counter

from prompts import (
    no_narrative_prompt, naive_narrative_prompt, compact_narrative_prompt,
    full_narrative, full_narrative_no_time, full_narrative_no_time_rnd,
    compact_no_time_prompt, compact_no_time_prompt_rnd
)

import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format="%(asctime)s: %(message)s")
logger = logging.getLogger(__name__)

def set_seed(seed_value=5550):
    # os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
    os.environ["PYTHONHASHSEED"] = str(seed_value)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    # torch.use_deterministic_algorithms(True)
    # torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

def parse_args():
    parser = argparse.ArgumentParser(description="Universal finetuning script for LLMs or MedBERT-like models.")
    parser.add_argument("--model_type", type=str, choices=["llm", "medbert"], default="llm",
                        help="Model type: llm o medbert")
    
    parser.add_argument("--model_name", type=str, required=True,
                        help="Name of the model from Hugging Face")
    
    parser.add_argument("--peft", action="store_true", help="Usa PEFT (solo per LLM)")

    parser.add_argument("--use_quantization", action="store_true",
                        help="Quantization 4 bit")
    
    parser.add_argument("--cache_dir", type=str, default="/root/MIMICIV/cache",
                        help="Directory for cache e saving models")
    
    parser.add_argument("--input_csv", type=str, default="/mnt/vdb/data/landmark_df_evo.csv",
                        help="Path to file CSV di input")
    
    parser.add_argument("--train_csv", type=str, default="/mnt/vdb/data/landmark_evo_train.csv",
                        help="Path to file CSV di training")
    
    parser.add_argument("--val_csv", type=str, default="/mnt/vdb/data/landmark_evo_vali.csv",
                        help="Path to file CSV di validation")
    
    parser.add_argument("--test_csv", type=str, default="/mnt/vdb/data/landmark_evo_test.csv",
                        help="Path to file CSV di test") # evo as well 
    
    parser.add_argument("--prompt_type", type=str, choices=["naive", "compact", "compact_no_time", "compact_narrative", "no", "no_set", "full", "full_no_time", "full_no_time_rnd"], default="compact",
                        help="Prompting type to use: 'naive', 'compact'  o 'no' (nessuna narrativa)")
    
    parser.add_argument("--max_visits", type=int, default=3,
                        help="number of visits to consider for each patient (max_visits)")
    
    parser.add_argument("--all_landmarks", action="store_true",
                        help="Process all landmark (from 1 to max_visits) or just the last one")
    
    parser.add_argument("--batch_size", type=int, default=8,
                        help="Batch size for DataLoader")
    
    parser.add_argument("--gradient_accumulation_steps", type=int, default=1,
                            help="Number of steps for gradient accumulation (useful for large models)")

    parser.add_argument("--epochs", type=int, default=20,
                        help="Num epochs for training")
    
    parser.add_argument("--patience", type=int, default=3,
                        help="Early stopping patience")
    
    parser.add_argument("--max_length", type=int, default=512,
                        help="Token Max lenght")
    
    parser.add_argument("--lr", type=float, default=2e-5,
                        help="Learning rate")
    
    parser.add_argument("--early", type=str, choices=["auc", "loss", "f1"], default="auc",
                        help="Early stopping criterion: 'auc', 'loss' or 'f1")
    
    parser.add_argument("--seed", type=int, default=9550,
                        help="Seed for riproducibility")

    parser.add_argument("--when_counting_death", type=str, choices=["last_visit", "landmark"], default="last_visit",
                    help="When counting death: 'last_visit' (considering the last visit) or 'landmark' (considering the current landmark visit)")
    
    args = parser.parse_args()

    # Log arguemnts values
    for arg, value in sorted(vars(args).items()):
        logger.info("Argument %s: %r", arg, value)

    return args


ImportError: cannot import name 'no_narrative_prompt_set' from 'prompts' (/root/MIMICIV/src/prompts.py)

In [2]:
def load_tokenizer(model_name, model_type, hf_token, cache_dir):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        token=hf_token if model_type == "llm" else None,
        cache_dir=cache_dir,
        trust_remote_code=True
    )
    return tokenizer

def load_model(model_name, model_type, tokenizer, cache_dir, hf_token, use_peft, use_quantization):
    if model_type == "llm":

        if use_quantization:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.bfloat16
            )
        else:
            bnb_config = None

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=2,
            torch_dtype=torch.bfloat16,
            device_map='auto',
            token=hf_token,
            cache_dir=cache_dir,
            quantization_config=None if use_quantization else bnb_config  # Use bnb for quantization
        )
        model.config.pad_token_id = tokenizer.eos_token_id
        if use_peft:
            # TODO: test different settings for LoraConfig
            lora_config = LoraConfig(
                r=8,
                lora_alpha=16,
                target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
                lora_dropout=0.1,
                bias='none',
                task_type="SEQ_CLS"
            )
            model = get_peft_model(model, lora_config)
            model.print_trainable_parameters()
    else:
        # For clinical models like MedBERT or similar
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=2,
            cache_dir=cache_dir
        )
    return model

class ClinicalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512): # depending on the model!! 
        self.encodings = tokenizer(
            texts, truncation=True, padding=True, max_length=max_length
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

def train_and_evaluate(model, train_loader, val_loader, args, landmark_visit):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)
    total_steps = args.epochs * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.06 * total_steps),
        num_training_steps=total_steps
    )
    best_model_path = get_best_model_path(args, landmark_visit)
    best_auc, best_f1, best_val_loss = 0.0, 0.0, float("inf")
    epochs_no_improve = 0
    
    logger.info(f"Early stopping criterion: {args.early}")
    
    for epoch in range(args.epochs):
        model.train()
        total_train_loss = 0
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader):
            inputs = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**inputs)
            loss = outputs.loss / args.gradient_accumulation_steps
            loss.backward()
            total_train_loss += loss.item()

            if (step + 1) % args.gradient_accumulation_steps == 0 or (step + 1) == len(train_loader):
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
        
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        total_val_loss = 0
        val_preds, val_labels = [], []

        with torch.no_grad():
            for batch in val_loader:
                inputs = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**inputs)
                val_loss = outputs.loss
                total_val_loss += val_loss.item()
                probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().float().numpy() # Try
                val_preds.extend(probs)
                val_labels.extend(batch['labels'].cpu().numpy())
                
        avg_val_loss = total_val_loss / len(val_loader)
        val_auc = roc_auc_score(val_labels, val_preds)

        # Compute F1-score
        threshold = 0.5
        val_preds_binary = (np.array(val_preds) >= threshold).astype(int)
        val_f1 = f1_score(val_labels, val_preds_binary, zero_division=0)

        logger.info(
            f"Landmark {landmark_visit} | Epoch {epoch+1}/{args.epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"Validation AUC: {val_auc:.4f} | "
            f"F1-Score: {val_f1:.4f}"
        )
        
        # Early stopping logic
        condition = False
        if args.early == "auc":
            condition = val_auc > best_auc
        elif args.early == "loss":
            condition = avg_val_loss < best_val_loss
        elif args.early == "f1":
            condition = val_f1 > best_f1
        else:
            raise ValueError("Invalid early stopping criterion. Use 'auc', 'loss' or 'f1'.")

        if condition == True:
            best_auc = val_auc
            best_f1 = val_f1
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_model_path)
            logger.info(f"New best model saved at {best_model_path} with AUC: {best_auc:.4f}, F1: {best_f1:.4f}, Val Loss: {best_val_loss:.4f}")
        else:
            epochs_no_improve += 1
            logger.info(f"No improvement in epoch {epoch+1}. Current best AUC: {best_auc:.4f}, F1: {best_f1:.4f}, Val Loss: {best_val_loss:.4f}. "
                        f"Epochs without improvement: {epochs_no_improve}/{args.patience}")
            if epochs_no_improve >= args.patience:
                logger.info(f"Early stopping triggered at epoch {epoch+1}. Best AUC: {best_auc:.4f}, F1: {best_f1:.4f}, Val Loss: {best_val_loss:.4f}")
                break

    model.load_state_dict(torch.load(best_model_path))
    return best_auc, best_model_path, best_f1, best_val_loss, epoch + 1

def get_best_model_path(args, landmark_visit):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_model_name = args.model_name.replace("/", "_")
    filename = f"best_model_{safe_model_name}_{args.seed}_landmark{landmark_visit}_{args.prompt_type}_{args.max_visits}_{args.max_length}_{args.when_counting_death}_all_landmarks_{args.all_landmarks}_{timestamp}_.pt"
    best_model_dir = os.path.join(args.cache_dir, 'best')
    os.makedirs(best_model_dir, exist_ok=True)
    return os.path.join(best_model_dir, filename)

def bootstrap_auc_ci(y_true, y_pred, n_bootstraps=1000, alpha=0.95, seed=42):
    bootstrapped_scores = []
    rng = np.random.RandomState(seed)
    for _ in range(n_bootstraps):
        indices = rng.randint(0, len(y_pred), len(y_pred))
        if len(np.unique(np.array(y_true)[indices])) < 2:
            continue
        score = roc_auc_score(np.array(y_true)[indices], np.array(y_pred)[indices])
        bootstrapped_scores.append(score)
    sorted_scores = np.array(bootstrapped_scores)
    sorted_scores.sort()
    lower = sorted_scores[int((1.0 - alpha) / 2 * len(sorted_scores))]
    upper = sorted_scores[int((alpha + (1.0 - alpha) / 2) * len(sorted_scores))]
    return lower, upper


NameError: name 'Dataset' is not defined

### Inizio del Main

In [ ]:
sys.argv = [''] + [
    '--model_type', 'medbert',
    '--model_name', 'answerdotai/ModernBERT-large', #    emilyalsentzer/Bio_ClinicalBERT  meta-llama/Llama-3.1-8B answerdotai/ModernBERT-large Charangan/MedBERT
    #'--peft',
    # '--use_quantization',
    '--cache_dir', '/root/MIMICIV/cache',
    '--input_csv', '/root/MIMICIV/src/landmark_df_evo_correct.csv',
    '--train_csv', '/root/MIMICIV/data/splitted/landmark_evo_train_dod.csv',
    '--val_csv', '/root/MIMICIV/data/splitted/landmark_evo_vali_dod.csv',
    '--test_csv', '/root/MIMICIV/data/splitted/landmark_evo_test_dod.csv',
    '--prompt_type', 'no_set',  # 'naive', 'compact', 'no', 'full', 'full_no_time', 'full_no_time_rnd'
    '--max_visits', '3',
    '--all_landmarks',
    '--batch_size', '8',
    '--epochs', '20',
    '--gradient_accumulation_steps', '1',
    '--patience', '3',
    '--max_length', '512',
    '--lr', '2e-5',
    '--early', 'loss',
    '--seed', '9550',
    '--when_counting_death', 'last_visit'
]

In [5]:
torch.cuda.empty_cache()
torch.cuda.is_available()

args = parse_args()

# Set seed for reproducibility
set_seed(args.seed)

hf_token = os.getenv("HF_TOKEN")

if args.model_type == "llm" and hf_token is None:
    raise ValueError("Set the HF_TOKEN environment variable for authentication.")
if args.model_type == "llm":
    login(hf_token)

tokenizer = load_tokenizer(args.model_name, args.model_type, hf_token, args.cache_dir)
model = load_model(args.model_name, args.model_type, tokenizer, args.cache_dir, hf_token, args.peft, args.use_quantization).to(device="cpu")

if tokenizer.pad_token is None:
    logger.warning("Tokenizer non ha un pad_token. Lo aggiungo manualmente come [PAD].")
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))
    
# After having resized the model, move it to the appropriate device    
device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Using device: {device}")
model.to(device)

# if re.search("evo.csv$", args.input_csv):
#     narrative_prompt = compact_narrative_prompt
# else:
#     narrative_prompt = naive_narrative_prompt
# logger.info(f"Using narrative prompt: {narrative_prompt.__name__}")

if args.prompt_type == "naive":
    narrative_prompt = naive_narrative_prompt
elif args.prompt_type == "compact":
    narrative_prompt = compact_narrative_prompt
elif args.prompt_type == "compact_no_time":
    narrative_prompt = compact_no_time_prompt
elif args.prompt_type == "compact_no_time_rnd":
    narrative_prompt = compact_no_time_prompt_rnd
elif args.prompt_type == "compact_narrative":
    narrative_prompt = compact_narrative_prompt
elif args.prompt_type == "no":
    narrative_prompt = no_narrative_prompt
elif args.prompt_type == "no_set":
    narrative_prompt = no_narrative_prompt_set
elif args.prompt_type == "full":
    narrative_prompt = full_narrative
elif args.prompt_type == "full_no_time":
    narrative_prompt = full_narrative_no_time
elif args.prompt_type == "full_no_time_rnd":
    narrative_prompt = full_narrative_no_time_rnd
else:
    raise ValueError("Invalid prompt type. Use 'naive' or 'compact'.")
logger.info(f"Using prompt type: {args.prompt_type}")

# Reading data
#landmark_df = pd.read_csv(args.input_csv, na_values=['', 'None', 'NaN', 'na', 'nan'])
#landmark_df = landmark_df.fillna('')
full_train_df = pd.read_csv(args.train_csv, na_values=['', 'None', 'NaN', 'na', 'nan']).fillna('')
full_val_df = pd.read_csv(args.val_csv, na_values=['', 'None', 'NaN', 'na', 'nan']).fillna('')
full_test_df = pd.read_csv(args.test_csv, na_values=['', 'None', 'NaN', 'na', 'nan']).fillna('')

# For testing problems in landmark/last_visit scenarios
full_train_df["correspondence"] = (full_train_df["death_in_90days"] == full_train_df["death_90days_landmark"]).astype(int)
full_test_df["correspondence"] = (full_test_df["death_in_90days"] == full_test_df["death_90days_landmark"]).astype(int)
full_val_df["correspondence"] = (full_val_df["death_in_90days"] == full_val_df["death_90days_landmark"]).astype(int)

# if args.prompt_type == 'full':
#     dataframe_no_overlap_2_merge = pd.read_csv('/root/MIMICIV/src/dataframe_no_overlap_2_merge.csv', na_values=['', 'None', 'NaN', 'na', 'nan']).fillna('')
#     full_train_df_no_overlap = full_train_df.merge(dataframe_no_overlap_2_merge, on=['subject_id', 'hadm_id'], how='inner')
#     full_val_df_no_overlap = full_val_df.merge(dataframe_no_overlap_2_merge, on=['subject_id', 'hadm_id'], how='inner')
#     full_test_df_no_overlap = full_test_df.merge(dataframe_no_overlap_2_merge, on=['subject_id', 'hadm_id'], how='inner')

#     full_train_df_no_overlap['days_since_last_visit'] = full_train_df_no_overlap['days_since_last_visit'].replace(0, 1)
#     full_val_df_no_overlap['days_since_last_visit'] = full_val_df_no_overlap['days_since_last_visit'].replace(0, 1)
#     full_test_df_no_overlap['days_since_last_visit'] = full_test_df_no_overlap['days_since_last_visit'].replace(0, 1)

#     full_train_df_no_overlap['days_since_previous_visit_cumulate'] = full_train_df_no_overlap.groupby('subject_id')['days_since_previous_visit'].transform(lambda x: [list(x[:i+1]) for i in range(len(x))])
#     full_train_df_no_overlap['days_since_previous_visit_cumulate_sum'] = full_train_df_no_overlap['days_since_previous_visit_cumulate'].apply(lambda x: np.cumsum([y for y in x if y > 0][::-1])[::-1] if isinstance(x, list) else [-1])

#     full_val_df_no_overlap['days_since_previous_visit_cumulate'] = full_val_df_no_overlap.groupby('subject_id')['days_since_previous_visit'].transform(lambda x: [list(x[:i+1]) for i in range(len(x))])
#     full_val_df_no_overlap['days_since_previous_visit_cumulate_sum'] = full_val_df_no_overlap['days_since_previous_visit_cumulate'].apply(lambda x: np.cumsum([y for y in x if y > 0][::-1])[::-1] if isinstance(x, list) else [-1])

#     full_test_df_no_overlap['days_since_previous_visit_cumulate'] = full_test_df_no_overlap.groupby('subject_id')['days_since_previous_visit'].transform(lambda x: [list(x[:i+1]) for i in range(len(x))])
#     full_test_df_no_overlap['days_since_previous_visit_cumulate_sum'] = full_test_df_no_overlap['days_since_previous_visit_cumulate'].apply(lambda x: np.cumsum([y for y in x if y > 0][::-1])[::-1] if isinstance(x, list) else [-1])

# Older version, now we are splitting the data in the splitting_data.py script
filtered_datasets = []
for dataset in [full_train_df, full_val_df, full_test_df]:
    visit_counts = dataset['subject_id'].value_counts()
    selected_patients = visit_counts[visit_counts == args.max_visits].index
    dataset_selected = dataset[dataset['subject_id'].isin(selected_patients)].copy()
    filtered_datasets.append(dataset_selected)

# visit_counts = landmark_df['subject_id'].value_counts()
# selected_patients = visit_counts[visit_counts == args.max_visits].index
# df_selected = landmark_df[landmark_df['subject_id'].isin(selected_patients)].copy()

train_df_selected, val_df_selected, test_df_selected = filtered_datasets

predictions_list, labels_list, results = [], [], []

if args.all_landmarks:
    logger.info(f"Processing all landmarks from 1 to {args.max_visits}")
    start = 1
    end = args.max_visits + 1
else:
    logger.info(f"Processing only the last landmark visit: {args.max_visits}")
    start = args.max_visits
    end = args.max_visits + 1


usage:  [-h] [--model_type {llm,medbert}] --model_name MODEL_NAME [--peft] [--use_quantization] [--cache_dir CACHE_DIR]
        [--input_csv INPUT_CSV] [--train_csv TRAIN_CSV] [--val_csv VAL_CSV] [--test_csv TEST_CSV]
        [--prompt_type {naive,compact,compact_no_time,compact_narrative,no,no_set,full,full_no_time,full_no_time_rnd}]
        [--max_visits MAX_VISITS] [--all_landmarks] [--batch_size BATCH_SIZE] [--gradient_accumulation_steps GRADIENT_ACCUMULATION_STEPS]
        [--epochs EPOCHS] [--patience PATIENCE] [--max_length MAX_LENGTH] [--lr LR] [--early {auc,loss,f1}] [--seed SEED]
        [--when_counting_death {last_visit,landmark}]
: error: argument --when_counting_death: invalid choice: 'last_visti' (choose from last_visit, landmark)


AttributeError: 'tuple' object has no attribute 'tb_frame'

In [ ]:


landmark_visit = 1 # Change this to the desired landmark visit
logger.info(f"Preparing data for Landmark {landmark_visit}")

# df_subset = df_selected[df_selected['landmark_visit'] == landmark_visit]
# patients = df_subset['subject_id'].unique()
# train_patients, test_patients = train_test_split(
#     patients, test_size=0.2, random_state=args.seed,
#     stratify=df_subset.groupby('subject_id')['death_in_90days'].max()
# )
train_df = train_df_selected[train_df_selected['landmark_visit'] == landmark_visit].copy()
val_df = val_df_selected[val_df_selected['landmark_visit'] == landmark_visit].copy()
test_df = test_df_selected[test_df_selected['landmark_visit'] == landmark_visit].copy()

In [7]:
import transformers
print(transformers.__version__)

4.52.3


In [8]:
train_df.head()
row = train_df.iloc[1]
row
(ast.literal_eval(row['days_since_last_visit_cumulate_sum'])[])

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2107543592.py, line 4)

#### Let's check the differences in terms of 90-days-after if considering the last-visit or the current-landmark as a temporal point.
We should have differences only on the 1st and 2nd landmark while on the 3rd we should have exactly the same.

In [9]:
(full_train_df["correspondence"].sum()/len(full_train_df))*100 # globale

np.float64(95.62927061814715)

##### Andiamo ad ispezionare caso per caso nel nostro subsample di tre visite...

In [13]:
(full_train_df[(full_train_df["num_total_visits"]==3) & (full_train_df["landmark_visit"]==1)]["correspondence"].sum()/len(full_train_df[(full_train_df["num_total_visits"]==3) & (full_train_df["landmark_visit"]==1)]))*100

88.32918739635157

In [14]:
(full_train_df[(full_train_df["num_total_visits"]==3) & (full_train_df["landmark_visit"]==2)]["correspondence"].sum()/len(full_train_df[(full_train_df["num_total_visits"]==3) & (full_train_df["landmark_visit"]==2)]))*100

92.74461028192371

In [16]:
(full_train_df[(full_train_df["num_total_visits"]==3) & (full_train_df["landmark_visit"]==3)]["correspondence"].sum()/len(full_train_df[(full_train_df["num_total_visits"]==3) & (full_train_df["landmark_visit"]==3)]))*100

100.0

##### Quick check on test and validation as well

In [17]:
# Test
(full_test_df[(full_test_df["num_total_visits"]==3) & (full_test_df["landmark_visit"]==1)]["correspondence"].sum()/len(full_test_df[(full_test_df["num_total_visits"]==3) & (full_test_df["landmark_visit"]==1)]))*100

88.65671641791046

In [18]:
100-88.66

11.340000000000003

In [19]:
# Validation
(full_val_df[(full_val_df["num_total_visits"]==3) & (full_val_df["landmark_visit"]==1)]["correspondence"].sum()/len(full_val_df[(full_val_df["num_total_visits"]==3) & (full_val_df["landmark_visit"]==1)]))*100

89.05472636815921

#### We should understand why on the 1st and 2nd landmark we get very different results from the "last_visit" scenario.
Is it due to an error in the code or to something else?
How should we investigate this?

A possible idea, just to get a more clear idea of the observed phenomenon, is to understand how many of the unmatched case dies in 90*3 days.

In [73]:
AAAA =(full_train_df[(full_train_df["num_total_visits"]==3) & (full_train_df["landmark_visit"]==1) & (full_train_df["correspondence"]==0)])

In [76]:
AAAA.columns

Index(['subject_id', 'hadm_id', 'admission_category', 'landmark_visit', 'age_at_landmark', 'gender', 'num_total_visits',
       'death_in_90days', 'med_text', 'diag_text', 'proc_text', 'dose_text', 'new_medications', 'new_diagnoses',
       'new_procedures', 'new_dose', 'no_more_diagnoses', 'no_more_medications', 'no_more_procedures', 'no_more_dose',
       'meds_per_visit', 'diag_per_visit', 'proc_per_visit', 'dose_per_visit', 'gender_numeric',
       'days_since_previous_visit', 'admittime', 'dischtime', 'days_until_next_visit', 'mins_until_next_visit',
       'days_since_last_visit', 'mins_since_last_visit', 'days_since_last_visit_cumulate',
       'days_since_last_visit_cumulate_sum', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod',
       'death_90days_landmark', 'correspondence'],
      dtype='object')

In [90]:
AAAA['delta_days'] = (pd.to_datetime(AAAA['dod']) - pd.to_datetime(AAAA['dischtime'])).dt.days

<positron-console-cell-90>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [97]:
len(AAAA[AAAA['delta_days'] < 365])/len(AAAA) * 100

38.24748371817643

In [98]:
len(AAAA[AAAA['delta_days'] < 270])/len(AAAA) * 100

30.669034931912375

In [99]:
len(AAAA[AAAA['delta_days'] < 180])/len(AAAA) * 100

20.485494375370042

#### Let's continue the process

In [12]:
train_texts = train_df.apply(narrative_prompt, axis=1).tolist()
val_texts = val_df.apply(narrative_prompt, axis=1).tolist()
test_texts = test_df.apply(narrative_prompt, axis=1).tolist()

if args.when_counting_death == "last_visit":
    train_labels = train_df['death_in_90days'].tolist()
    val_labels = val_df['death_in_90days'].tolist()
    test_labels = test_df['death_in_90days'].tolist()
elif args.when_counting_death == "landmark":
    train_labels = train_df['death_90days_landmark'].tolist()
    val_labels = val_df['death_90days_landmark'].tolist()
    test_labels = test_df['death_90days_landmark'].tolist()

# Media di lunghezza dei testi
avg_train_length = np.mean([len(text.split()) for text in train_texts])
avg_val_length = np.mean([len(text.split()) for text in val_texts])
avg_test_length = np.mean([len(text.split()) for text in test_texts])
logger.info(f"Average train text length: {avg_train_length:.2f} words")
logger.info(f"Average validation text length: {avg_val_length:.2f} words")      
logger.info(f"Average test text length: {avg_test_length:.2f} words")
logger.info(f"Training on {len(train_texts)} samples, validating on {len(val_texts)}, testing on {len(test_texts)} samples")


2025-08-06 07:00:21,650: Average train text length: 332.75 words
2025-08-06 07:00:21,651: Average validation text length: 334.67 words
2025-08-06 07:00:21,652: Average test text length: 332.69 words
2025-08-06 07:00:21,653: Training on 14472 samples, validating on 1608, testing on 4020 samples


In [16]:
train_df['death_in_90days'].tolist()[0:10]

[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]

In [14]:
train_df['correspondence'].tolist()[0:10]

[1, 1, 1, 0, 1, 1, 1, 1, 1, 1]

In [17]:
train_labels[0:10]

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [ ]:

from transformers import AutoTokenizer
import matplotlib.pyplot as plt

def compute_truncation_stats(texts, tokenizer, max_length=512, show_plots=True):
    token_lengths = []
    truncation_amounts = []
    word_truncation_amounts = []

    for text in texts:
        # Tokenization without truncation
        tokens_full = tokenizer.encode(text, truncation=False)
        tokens_truncated = tokenizer.encode(text, truncation=True, max_length=max_length)

        token_lengths.append(len(tokens_full))

        if len(tokens_full) > max_length:
            truncation_amounts.append(len(tokens_full) - max_length)

            # Word-level truncation estimation
            original_word_count = len(text.split())
            truncated_text = tokenizer.decode(tokens_truncated, skip_special_tokens=True)
            truncated_word_count = len(truncated_text.split())

            word_diff = original_word_count - truncated_word_count
            word_truncation_amounts.append(word_diff)

    total = len(texts)
    truncated = len(truncation_amounts)

    print(f"Total samples: {total}")
    print(f"Truncated samples: {truncated} ({truncated / total * 100:.2f}%)")

    if truncation_amounts:
        print(f"\n--- Token Truncation ---")
        print(f"Avg tokens truncated: {sum(truncation_amounts) / truncated:.2f}")
        print(f"Max tokens truncated: {max(truncation_amounts)}")

        print(f"\n--- Word Truncation Estimate ---")
        print(f"Avg words truncated: {sum(word_truncation_amounts) / truncated:.2f}")
        print(f"Max words truncated: {max(word_truncation_amounts)}")

        if show_plots:
            # Plot token truncation
            plt.hist(truncation_amounts, bins=30)
            plt.title("Distribution of Truncated Tokens")
            plt.xlabel("Tokens truncated")
            plt.ylabel("Number of samples")
            plt.show()

            # Plot word truncation
            plt.hist(word_truncation_amounts, bins=30)
            plt.title("Estimated Distribution of Truncated Words")
            plt.xlabel("Words truncated")
            plt.ylabel("Number of samples")
            plt.show()
    else:
        print("No samples were truncated.")


compute_truncation_stats(train_texts, tokenizer, max_length=args.max_length)

In [58]:
print(train_texts[0])

Based on this information, what is the probability of mortality within 90 days?
Diagnoses: Other constipation; Other ascites; Other and unspecified alcohol dependence, continuous; Unspecified deficiency anemia; Hypopotassemia; Acute alcoholic hepatitis; Portal hypertension; Pneumonia, organism unspecified; Urinary tract infection, site not specified; Other specified forms of effusion, except tuberculous; Alcoholic cirrhosis of liver; Anxiety state, unspecified; Unspecified protein-calorie malnutrition; Hyposmolality and/or hyponatremia
Medications: Phytonadione; SW; Furosemide; Heparin; Multivitamins; Lidocaine 5% Patch; Lorazepam; Multivitamin IV; 0.9% Sodium Chloride; FoLIC Acid; Senna; Acetaminophen; Albumin 25% (12.5g / 50mL); Thiamine; Ondansetron; OxycoDONE (Immediate Release) ; Potassium Chloride; Ciprofloxacin HCl; Neutra-Phos; 1/2 NS; Nicotine Patch; Magnesium Sulfate; Morphine Sulfate; Sodium Chloride 0.9%  Flush; Docusate Sodium
Procedures: 1 mg of Lorazepam in mL through IV

In [57]:
train_dataset = ClinicalDataset(train_texts, train_labels, tokenizer, max_length=args.max_length)
val_dataset = ClinicalDataset(val_texts, val_labels, tokenizer, max_length=args.max_length)
test_dataset = ClinicalDataset(test_texts, test_labels, tokenizer, max_length=args.max_length)
train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)

logger.info(f"Fine-tuning at Landmark {landmark_visit}")
start_time = datetime.datetime.now()

2025-08-05 10:22:11,806: Fine-tuning at Landmark 1


In [105]:
args.patience = 5
args.early = "loss"

In [100]:
# torch.set_float32_matmul_precision('high') 
best_auc, best_model_path, best_val_f1, best_val_loss, epochs_done = train_and_evaluate(
    model, train_loader, val_loader, args, landmark_visit
)
elapsed_time = datetime.datetime.now() - start_time
elapsed_seconds = elapsed_time.total_seconds()
logger.info(f"Tempo impiegato per Landmark {landmark_visit}: {elapsed_seconds:.1f} secondi")
logger.info(f"Tempo medio per epoca: {elapsed_seconds / epochs_done:.1f} secondi") # This is the important one

# VALUTAZIONE FINALE SUL TEST SET
model.eval()
test_preds, test_labels = [], []
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

with torch.no_grad():
    for batch in test_loader:
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().float().numpy() # on cpu for sklearn metrics
        test_preds.extend(probs)
        test_labels.extend(batch['labels'].cpu().float().numpy()) # on cpu for sklearn metrics

test_auc = roc_auc_score(test_labels, test_preds)
test_f1 = f1_score(test_labels, np.array(test_preds) >= 0.5)

logger.info(
    f"Final Test AUC (Landmark {landmark_visit}): {test_auc:.4f} | "
    f"Test F1-Score: {test_f1:.4f}"
)

results.append({
    'Landmark': landmark_visit,
    'Val AUC': f"{best_auc:.4f}",
    'Val F1-Score': f"{best_val_f1:.4f}",
    'Test AUC': f"{test_auc:.4f}",
    'Test F1-Score': f"{test_f1:.4f}",
    'Patients (Test)': len(test_df),
    'Model Path': best_model_path,
    'Validation Loss': f"{best_val_loss:.4f}",
    'Training Time (s)': f"{elapsed_seconds:.1f}"
})

2025-08-05 13:40:24,449: Early stopping criterion: f1


Using device: cuda


2025-08-05 13:45:29,634: Landmark 1 | Epoch 1/20 | Train Loss: 0.0402 | Val Loss: 0.1472 | Validation AUC: 0.8741 | F1-Score: 0.1644
2025-08-05 13:50:34,659: Landmark 1 | Epoch 2/20 | Train Loss: 0.0405 | Val Loss: 0.2346 | Validation AUC: 0.7776 | F1-Score: 0.2245
2025-08-05 13:55:41,061: Landmark 1 | Epoch 3/20 | Train Loss: 0.0347 | Val Loss: 0.2270 | Validation AUC: 0.7014 | F1-Score: 0.2410
2025-08-05 14:00:46,907: Landmark 1 | Epoch 4/20 | Train Loss: 0.0306 | Val Loss: 0.1847 | Validation AUC: 0.7215 | F1-Score: 0.2250
2025-08-05 14:05:51,260: Landmark 1 | Epoch 5/20 | Train Loss: 0.0183 | Val Loss: 0.2082 | Validation AUC: 0.7137 | F1-Score: 0.2895
2025-08-05 14:10:57,090: Landmark 1 | Epoch 6/20 | Train Loss: 0.0139 | Val Loss: 0.2975 | Validation AUC: 0.6918 | F1-Score: 0.1493
2025-08-05 14:16:01,021: Landmark 1 | Epoch 7/20 | Train Loss: 0.0136 | Val Loss: 0.2483 | Validation AUC: 0.7604 | F1-Score: 0.2535
2025-08-05 14:21:06,222: Landmark 1 | Epoch 8/20 | Train Loss: 0.0092

KeyboardInterrupt: 

In [16]:
import torch
print(torch.__version__) 

2.4.1+cu121


In [22]:
narrative_prompt

<function prompts.no_narrative_prompt_set(row)>

In [ ]:

def no_narrative_prompt_set(row, to_split="\n"):
    narrative = 'Based on this information, what is the probability of mortality within 90 days?'
    if pd.notna(row['diag_text']) and row['diag_text'].strip():
        narrative += f'\nDiagnoses: {"; ".join(set(row["diag_text"].split(f"{to_split}")))}'
    if pd.notna(row['med_text']) and row['med_text'].strip():
        narrative += f'\nMedications: {"; ".join(set(row["med_text"].split(f"{to_split}")))}'
    if pd.notna(row['proc_text']) and row['proc_text'].strip():
        narrative += f'\nProcedures: {"; ".join(set(row["proc_text"].split(f"{to_split}")))}'
    return narrative

In [18]:
row = train_df_selected.iloc[0]
f"diagnosi: {'; '.join(set(row['diag_text'].split(split2)))}"

NameError: name 'split2' is not defined

In [66]:
narrative = 'Based on this information, what is the probability of mortality within 90 days?'


In [69]:
if pd.notna(row['diag_text']) and row['diag_text'].strip():
    print("TRUE")

TRUE


In [80]:
split2 = '\n'

In [85]:
(row['diag_text'].split(f"{split2}"))

['Alcoholic cirrhosis of liver',
 'Pneumonia, organism unspecified',
 'Other ascites',
 'Portal hypertension',
 'Urinary tract infection, site not specified',
 'Unspecified protein-calorie malnutrition',
 'Hyposmolality and/or hyponatremia',
 'Other specified forms of effusion, except tuberculous',
 'Acute alcoholic hepatitis',
 'Hypopotassemia',
 'Other and unspecified alcohol dependence, continuous',
 'Other constipation',
 'Unspecified deficiency anemia',
 'Anxiety state, unspecified']

In [86]:
f"Diagnoses: {'; '.join(set(row['diag_text'].split(f'{split2}')))}"

'Diagnoses: Anxiety state, unspecified; Other constipation; Portal hypertension; Hypopotassemia; Pneumonia, organism unspecified; Other ascites; Acute alcoholic hepatitis; Unspecified protein-calorie malnutrition; Unspecified deficiency anemia; Hyposmolality and/or hyponatremia; Other specified forms of effusion, except tuberculous; Alcoholic cirrhosis of liver; Other and unspecified alcohol dependence, continuous; Urinary tract infection, site not specified'

In [19]:
print(no_narrative_prompt_set(row))

Based on this information, what is the probability of mortality within 90 days?
Diagnoses: Other and unspecified alcohol dependence, continuous; Alcoholic cirrhosis of liver; Unspecified protein-calorie malnutrition; Other specified forms of effusion, except tuberculous; Acute alcoholic hepatitis; Other constipation; Hyposmolality and/or hyponatremia; Anxiety state, unspecified; Pneumonia, organism unspecified; Hypopotassemia; Other ascites; Unspecified deficiency anemia; Portal hypertension; Urinary tract infection, site not specified
Medications: Neutra-Phos; 0.9% Sodium Chloride; Potassium Chloride; Magnesium Sulfate; Nicotine Patch; Docusate Sodium; Lidocaine 5% Patch; Furosemide; Multivitamin IV; Thiamine; Sodium Chloride 0.9%  Flush; SW; Heparin; Phytonadione; Ciprofloxacin HCl; Ondansetron; OxycoDONE (Immediate Release) ; Albumin 25% (12.5g / 50mL); Senna; 1/2 NS; FoLIC Acid; Acetaminophen; Lorazepam; Morphine Sulfate; Multivitamins
Procedures: 1 mg of Lorazepam in TAB through P

In [99]:
narrative_prompt = no_narrative_prompt_set 

In [100]:
logger.info(f"Preparing data for Landmark {landmark_visit}")
    
# df_subset = df_selected[df_selected['landmark_visit'] == landmark_visit]
# patients = df_subset['subject_id'].unique()
# train_patients, test_patients = train_test_split(
#     patients, test_size=0.2, random_state=args.seed,
#     stratify=df_subset.groupby('subject_id')['death_in_90days'].max()
# )
train_df = train_df_selected[train_df_selected['landmark_visit'] == landmark_visit].copy()
val_df = val_df_selected[val_df_selected['landmark_visit'] == landmark_visit].copy()
test_df = test_df_selected[test_df_selected['landmark_visit'] == landmark_visit].copy()

train_texts = train_df.apply(narrative_prompt, axis=1).tolist()
val_texts = val_df.apply(narrative_prompt, axis=1).tolist()
test_texts = test_df.apply(narrative_prompt, axis=1).tolist()
train_labels = train_df['death_in_90days'].tolist()
val_labels = val_df['death_in_90days'].tolist()
test_labels = test_df['death_in_90days'].tolist()

# Media di lunghezza dei testi
avg_train_length = np.mean([len(text.split()) for text in train_texts])
avg_val_length = np.mean([len(text.split()) for text in val_texts])
avg_test_length = np.mean([len(text.split()) for text in test_texts])
logger.info(f"Average train text length: {avg_train_length:.2f} words")
logger.info(f"Average validation text length: {avg_val_length:.2f} words")      
logger.info(f"Average test text length: {avg_test_length:.2f} words")
logger.info(f"Training on {len(train_texts)} samples, validating on {len(val_texts)}, testing on {len(test_texts)} samples")

train_dataset = ClinicalDataset(train_texts, train_labels, tokenizer, max_length=args.max_length)
val_dataset = ClinicalDataset(val_texts, val_labels, tokenizer, max_length=args.max_length)
test_dataset = ClinicalDataset(test_texts, test_labels, tokenizer, max_length=args.max_length)
train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)

logger.info(f"Fine-tuning at Landmark {landmark_visit}")
start_time = datetime.datetime.now()
best_auc, best_model_path, best_val_f1, best_val_loss, epochs_done = train_and_evaluate(
    model, train_loader, val_loader, args, landmark_visit
)
elapsed_time = datetime.datetime.now() - start_time
elapsed_seconds = elapsed_time.total_seconds()
logger.info(f"Tempo impiegato per Landmark {landmark_visit}: {elapsed_seconds:.1f} secondi")
logger.info(f"Tempo medio per epoca: {elapsed_seconds / epochs_done:.1f} secondi") # This is the important one

# VALUTAZIONE FINALE SUL TEST SET
model.eval()
test_preds, test_labels = [], []
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

with torch.no_grad():
    for batch in test_loader:
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().float().numpy() # on cpu for sklearn metrics
        test_preds.extend(probs)
        test_labels.extend(batch['labels'].cpu().float().numpy()) # on cpu for sklearn metrics


2025-07-30 09:18:46,604: Preparing data for Landmark 3
2025-07-30 09:18:48,691: Average train text length: 777.21 words
2025-07-30 09:18:48,693: Average validation text length: 773.40 words
2025-07-30 09:18:48,694: Average test text length: 774.63 words
2025-07-30 09:18:48,696: Training on 14472 samples, validating on 1608, testing on 4020 samples
2025-07-30 09:19:53,026: Fine-tuning at Landmark 3
2025-07-30 09:19:53,033: Early stopping criterion: loss


Using device: cuda


2025-07-30 09:27:50,614: Landmark 3 | Epoch 1/20 | Train Loss: 0.2836 | Val Loss: 0.2185 | Validation AUC: 0.9368 | F1-Score: 0.6495
2025-07-30 09:35:48,519: Landmark 3 | Epoch 2/20 | Train Loss: 0.2271 | Val Loss: 0.2059 | Validation AUC: 0.9416 | F1-Score: 0.6735
2025-07-30 09:43:47,320: Landmark 3 | Epoch 3/20 | Train Loss: 0.2073 | Val Loss: 0.2007 | Validation AUC: 0.9448 | F1-Score: 0.6959
2025-07-30 09:51:46,355: Landmark 3 | Epoch 4/20 | Train Loss: 0.1849 | Val Loss: 0.2110 | Validation AUC: 0.9482 | F1-Score: 0.7143
2025-07-30 09:59:44,865: Landmark 3 | Epoch 5/20 | Train Loss: 0.1549 | Val Loss: 0.2616 | Validation AUC: 0.9375 | F1-Score: 0.6898
2025-07-30 10:07:43,298: Landmark 3 | Epoch 6/20 | Train Loss: 0.1194 | Val Loss: 0.2600 | Validation AUC: 0.9343 | F1-Score: 0.6485
2025-07-30 10:07:43,300: Early stopping triggered at epoch 6
<positron-console-cell-100>:170: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses 

In [ ]:

test_auc = roc_auc_score(test_labels, test_preds)
test_f1 = f1_score(test_labels, np.array(test_preds) >= 0.5)

logger.info(
    f"Final Test AUC (Landmark {landmark_visit}): {test_auc:.4f} | "
    f"Test F1-Score: {test_f1:.4f}"
)

results.append({
    'Landmark': landmark_visit,
    'Val AUC': f"{best_auc:.4f}",
    'Val F1-Score': f"{best_val_f1:.4f}",
    'Test AUC': f"{test_auc:.4f}",
    'Test F1-Score': f"{test_f1:.4f}",
    'Patients (Test)': len(test_df),
    'Model Path': best_model_path,
    'Validation Loss': f"{best_val_loss:.4f}",
    'Training Time (s)': f"{elapsed_seconds:.1f}"
})

In [101]:
logger.info(
    f"Final Test AUC (Landmark {landmark_visit}): {test_auc:.4f} | "
    f"Test F1-Score: {test_f1:.4f}"
)

2025-08-05 14:25:27,432: Final Test AUC (Landmark 1): 0.8634 | Test F1-Score: 0.0000


In [103]:
print(start, end)

1 4


In [20]:
for landmark_visit in range(start, end):
    logger.info(f"Preparing data for Landmark {landmark_visit}")
    
    # df_subset = df_selected[df_selected['landmark_visit'] == landmark_visit]
    # patients = df_subset['subject_id'].unique()
    # train_patients, test_patients = train_test_split(
    #     patients, test_size=0.2, random_state=args.seed,
    #     stratify=df_subset.groupby('subject_id')['death_in_90days'].max()
    # )

    # Set the seed for reproducibility (for each landmark visit)
    set_seed(args.seed)

    # First, clean
    if 'model' in globals():
        del model
    if 'tokenizer' in globals():
        del tokenizer

    torch.cuda.empty_cache()

    # Load model and tokenizer
    hf_token = os.getenv("HF_TOKEN")
    if args.model_type == "general" and hf_token is None:
        raise ValueError("Set the HF_TOKEN environment variable for authentication.")
    if args.model_type == "general":
        login(hf_token)

    tokenizer = load_tokenizer(args.model_name, args.model_type, hf_token, args.cache_dir)    
    model = load_model(args.model_name, args.model_type, tokenizer, args.cache_dir, hf_token, args.peft, args.use_quantization).to(device="cpu")

    if tokenizer.pad_token is None:
        logger.warning("Tokenizer non ha un pad_token. Lo aggiungo manualmente come [PAD].")
        tokenizer.add_special_tokens({'pad_token': '[PAD]'})
        model.resize_token_embeddings(len(tokenizer))
        
    # After having resized the model, move it to the appropriate device    
    model.to("cuda" if torch.cuda.is_available() else "cpu")

    train_df = train_df_selected[train_df_selected['landmark_visit'] == landmark_visit].copy()
    val_df = val_df_selected[val_df_selected['landmark_visit'] == landmark_visit].copy()
    test_df = test_df_selected[test_df_selected['landmark_visit'] == landmark_visit].copy()

    train_texts = train_df.apply(narrative_prompt, axis=1).tolist()
    val_texts = val_df.apply(narrative_prompt, axis=1).tolist()
    test_texts = test_df.apply(narrative_prompt, axis=1).tolist()
    train_labels = train_df['death_in_90days'].tolist()
    val_labels = val_df['death_in_90days'].tolist()
    test_labels = test_df['death_in_90days'].tolist()

    # Media di lunghezza dei testi
    avg_train_length = np.mean([len(text.split()) for text in train_texts])
    avg_val_length = np.mean([len(text.split()) for text in val_texts])
    avg_test_length = np.mean([len(text.split()) for text in test_texts])
    logger.info(f"Average train text length: {avg_train_length:.2f} words")
    logger.info(f"Average validation text length: {avg_val_length:.2f} words")      
    logger.info(f"Average test text length: {avg_test_length:.2f} words")
    logger.info(f"Training on {len(train_texts)} samples, validating on {len(val_texts)}, testing on {len(test_texts)} samples")

    train_dataset = ClinicalDataset(train_texts, train_labels, tokenizer, max_length=args.max_length)
    val_dataset = ClinicalDataset(val_texts, val_labels, tokenizer, max_length=args.max_length)
    test_dataset = ClinicalDataset(test_texts, test_labels, tokenizer, max_length=args.max_length)
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=args.batch_size, shuffle=False)
    
    logger.info(f"Fine-tuning at Landmark {landmark_visit}")
    start_time = datetime.datetime.now()
    best_auc, best_model_path, best_val_f1, best_val_loss, epochs_done = train_and_evaluate(
        model, train_loader, val_loader, args, landmark_visit
    )
    elapsed_time = datetime.datetime.now() - start_time
    elapsed_seconds = elapsed_time.total_seconds()
    logger.info(f"Tempo impiegato per Landmark {landmark_visit}: {elapsed_seconds:.1f} secondi")
    logger.info(f"Tempo medio per epoca: {elapsed_seconds / epochs_done:.1f} secondi") # This is the important one

    # VALUTAZIONE FINALE SUL TEST SET
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    logger.info(f"Loading best model from {best_model_path} for final evaluation on test set")
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()
    test_preds, test_labels = [], []

    with torch.no_grad():
        for batch in test_loader:
            inputs = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().float().numpy() # on cpu for sklearn metrics
            test_preds.extend(probs)
            test_labels.extend(batch['labels'].cpu().float().numpy()) # on cpu for sklearn metrics

    test_auc = roc_auc_score(test_labels, test_preds)
    test_f1 = f1_score(test_labels, np.array(test_preds) >= 0.5)

    logger.info(
        f"Final Test AUC (Landmark {landmark_visit}): {test_auc:.4f} | "
        f"Test F1-Score: {test_f1:.4f}"
    )

    results.append({
        'Landmark': landmark_visit,
        'Val AUC': f"{best_auc:.4f}",
        'Val F1-Score': f"{best_val_f1:.4f}",
        'Test AUC': f"{test_auc:.4f}",
        'Test F1-Score': f"{test_f1:.4f}",
        'Patients (Test)': len(test_df),
        'Model Path': best_model_path,
        'Validation Loss': f"{best_val_loss:.4f}",
        'Training Time (s)': f"{elapsed_seconds:.1f}"
    })


results_df = pd.DataFrame(results)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
model_tag = args.model_name.replace("/", "_")
peft_tag = "_peft" if args.peft else ""
output_filename = (
    f"results_{args.model_type}_{model_tag}_visits{args.max_visits}{peft_tag}_{args.seed}_{args.prompt_type}_{timestamp}.csv"
)
logger.info(f"Saving results to {output_filename}")
results_df.to_csv(os.path.join(args.cache_dir, output_filename), index=False)

print(results_df)


2025-08-06 07:03:11,657: Preparing data for Landmark 1
Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2025-08-06 07:03:13,549: Average train text length: 332.75 words
2025-08-06 07:03:13,550: Average validation text length: 334.67 words
2025-08-06 07:03:13,551: Average test text length: 332.69 words
2025-08-06 07:03:13,552: Training on 14472 samples, validating on 1608, testing on 4020 samples
2025-08-06 07:03:36,873: Fine-tuning at Landmark 1
2025-08-06 07:03:36,878: Early stopping criterion: loss


Using device: cuda


/root/miniforge3/lib/python3.12/site-packages/torch/_inductor/compile_fx.py:194: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


AttributeError: 'Namespace' object has no attribute 'gradient_accumulation_steps'